In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/abraham/uni/ikt453/project/v1

/home/abraham/uni/ikt453/project/v1


In [2]:
import re
import os
import pandas as pd
from src.utils import disk
from itertools import chain

from src.clients import connect_mysql

In [3]:
season_dir = 'data/season'
season_dir = disk.listdir(season_dir)
u_game_ids = disk.read_json('data/unique_game_ids.json')
u_game_ids = u_game_ids[-1]['data']
u_game_ids = [v['game_id'] for v in u_game_ids]

In [4]:
import os
import re
from itertools import chain
import pandas as pd

def read_season(path: str):
    date_season_pattern = re.compile(r'games_(\d{4}-\d{2})_(\w+)')
    match = date_season_pattern.match(os.path.basename(path))
    season_label, season_type = match.groups()

    annotate = lambda x: {
        **x,
        'season_label': season_label,
        'season_type': season_type,
    }

    return map(annotate, disk.read_json(path))

def matchup_parser(x: str) -> tuple[str, str] | tuple[None, None]:
    if '@' in x:
        away_team, home_team = map(str.strip, x.split('@'))
        return home_team, away_team
    elif 'vs.' in x:
        home_team, away_team = map(str.strip, x.split('vs.'))
        return home_team, away_team
    return None, None

def normalize_season_matchup(subset: pd.DataFrame) -> pd.Series:
    subset = subset.copy()

    matchup_values = subset['matchup'].dropna().unique()

    home_team, away_team = matchup_parser(matchup_values[0])
    assert home_team is not None and away_team is not None, f"Could not parse matchup: {matchup_values[0]}"

    game_ids = subset['game_id'].unique()
    game_dates = subset['game_date'].unique()
    season_labels = subset['season_label'].unique()
    season_types = subset['season_type'].unique()

    assert len(game_ids) == 1
    assert len(game_dates) == 1
    assert len(season_labels) == 1
    assert len(season_types) == 1

    # map abbreviation -> team_id
    abbreviation2id = (
        subset[['team_abbreviation', 'team_id']]
        .drop_duplicates()
        .set_index('team_abbreviation')['team_id']
        .to_dict()
    )

    return pd.Series({
        'game_id': game_ids[0],
        'game_date': game_dates[0],
        'home_team': home_team,
        'away_team': away_team,
        'home_team_id': abbreviation2id.get(home_team),
        'away_team_id': abbreviation2id.get(away_team),
        'season_label': season_labels[0],
        'season_type': season_types[0],
    })

season_flat = chain.from_iterable(map(read_season, season_dir))
season_flat = pd.DataFrame(list(season_flat))
season_flat.columns = season_flat.columns.str.lower()
season_data = (
    season_flat
    .groupby('game_id')
    [season_flat.columns]
    .apply(normalize_season_matchup)
    .sort_values(by='game_date')
    .reset_index(drop=True)
)

In [5]:
season_data

,game_id,game_date,home_team,away_team,home_team_id,away_team_id,season_label,season_type
0,0020000008,2000-10-31,HOU,MIN,1.610613e+09,1.610613e+09,2000-01,Regular_Season
1,0020000001,2000-10-31,NYK,PHI,1.610613e+09,1.610613e+09,2000-01,Regular_Season
2,0020000002,2000-10-31,NJN,CLE,1.610613e+09,1.610613e+09,2000-01,Regular_Season
3,0020000003,2000-10-31,ORL,WAS,1.610613e+09,1.610613e+09,2000-01,Regular_Season
4,0020000004,2000-10-31,ATL,CHH,1.610613e+09,1.610613e+09,2000-01,Regular_Season
...,...,...,...,...,...,...,...,...
35341,0022501143,2026-04-06,ATL,NYK,1.610613e+09,1.610613e+09,2025-26,Regular_Season
35342,0022501144,2026-04-06,ORL,DET,1.610613e+09,1.610613e+09,2025-26,Regular_Season
35343,0022501145,2026-04-06,MEM,CLE,1.610613e+09,1.610613e+09,2025-26,Regular_Season
35344,0022501146,2026-04-06,SAS,PHI,1.610613e+09,1.610613e+09,2025-26,Regular_Season


In [6]:
conn = connect_mysql()
players = pd.read_sql("SELECT game_id, player_id FROM fact_player_game_stats", conn)

/tmp/ipykernel_444377/4162596801.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  players = pd.read_sql("SELECT game_id, player_id FROM fact_player_game_stats", conn)


In [7]:
players.shape, season_data.shape

((829073, 2), (35346, 8))

In [8]:
set_a = set(players.game_id)
set_b = set(season_data.game_id)

diff = set_a.difference(set_b)
same = set_a.intersection(set_b)
len(diff), len(same), len(set_a), len(set_b)

(42, 30659, 30701, 35346)

In [9]:
pgs = pd.merge(players, season_data, on='game_id', how='left')
pgs

,game_id,player_id,game_date,home_team,away_team,home_team_id,away_team_id,season_label,season_type
0,0020301147,1740,2004-04-10,SEA,DAL,1.610613e+09,1.610613e+09,2003-04,Regular_Season
1,0020301147,1738,2004-04-10,SEA,DAL,1.610613e+09,1.610613e+09,2003-04,Regular_Season
2,0020301147,958,2004-04-10,SEA,DAL,1.610613e+09,1.610613e+09,2003-04,Regular_Season
3,0020301147,951,2004-04-10,SEA,DAL,1.610613e+09,1.610613e+09,2003-04,Regular_Season
4,0020301147,2557,2004-04-10,SEA,DAL,1.610613e+09,1.610613e+09,2003-04,Regular_Season
...,...,...,...,...,...,...,...,...,...
829068,0021000263,201154,2010-12-01,TOR,WAS,1.610613e+09,1.610613e+09,2010-11,Regular_Season
829069,0021000263,201156,2010-12-01,TOR,WAS,1.610613e+09,1.610613e+09,2010-11,Regular_Season
829070,0021000263,202338,2010-12-01,TOR,WAS,1.610613e+09,1.610613e+09,2010-11,Regular_Season
829071,0021000263,201858,2010-12-01,TOR,WAS,1.610613e+09,1.610613e+09,2010-11,Regular_Season


In [10]:
draft = disk.read_json('data/samples_nbaapi/drafthistory.json')
draft = draft['resultSets'][0]
draft = pd.DataFrame(draft['rowSet'], columns=draft['headers'])
draft.columns = draft.columns.str.lower()

In [11]:
draft.head()

,person_id,player_name,season,round_number,round_pick,overall_pick,draft_type,team_id,team_city,team_name,team_abbreviation,organization,organization_type,player_profile_flag
0,1642843,Cooper Flagg,2025,1,1,1,Draft,1610612742,Dallas,Mavericks,DAL,Duke,College/University,1
1,1642844,Dylan Harper,2025,1,2,2,Draft,1610612759,San Antonio,Spurs,SAS,Rutgers,College/University,1
2,1642845,VJ Edgecombe,2025,1,3,3,Draft,1610612755,Philadelphia,76ers,PHI,Baylor,College/University,1
3,1642851,Kon Knueppel,2025,1,4,4,Draft,1610612766,Charlotte,Hornets,CHA,Duke,College/University,1
4,1642846,Ace Bailey,2025,1,5,5,Draft,1610612762,Utah,Jazz,UTA,Rutgers,College/University,1


In [23]:
drafted_players = draft[draft.round_number != 0]

In [27]:
set_a = set(players['player_id'])
set_b = set(drafted_players['person_id'].astype(str))

diff = set_a.difference(set_b)
same = set_a.intersection(set_b)
len(diff), len(same), len(set_a), len(set_b)

(2963, 1728, 4691, 7741)

In [28]:
player_ids = set_a | set_b
player_ids = list(player_ids)
# disk.write_json('data/unique_player_ids.json', player_ids)
len(player_ids)

10704

In [14]:
m = pgs.player_id.isin(diff)
pgs[m]

,game_id,player_id,game_date,home_team,away_team,home_team_id,away_team_id,season_label,season_type
8,0020301147,2499,2004-04-10,SEA,DAL,1.610613e+09,1.610613e+09,2003-04,Regular_Season
10,0020301147,2501,2004-04-10,SEA,DAL,1.610613e+09,1.610613e+09,2003-04,Regular_Season
15,0020301147,2605,2004-04-10,SEA,DAL,1.610613e+09,1.610613e+09,2003-04,Regular_Season
20,0020301147,281,2004-04-10,SEA,DAL,1.610613e+09,1.610613e+09,2003-04,Regular_Season
34,0011300011,203543,2013-10-07,BOS,TOR,1.610613e+09,1.610613e+09,2013-14,Pre_Season
...,...,...,...,...,...,...,...,...,...
829044,0011900044,203584,2019-10-12,BKN,LAL,1.610613e+09,1.610613e+09,2019-20,Pre_Season
829048,0011900044,204065,2019-10-12,BKN,LAL,1.610613e+09,1.610613e+09,2019-20,Pre_Season
829053,0021000263,101181,2010-12-01,TOR,WAS,1.610613e+09,1.610613e+09,2010-11,Regular_Season
829061,0021000263,202087,2010-12-01,TOR,WAS,1.610613e+09,1.610613e+09,2010-11,Regular_Season


In [15]:
u_game_ids = pd.Series(u_game_ids)

In [16]:
set_a = set(season_flat.GAME_ID.values)
set_b = set(u_game_ids.values)

AttributeError: 'DataFrame' object has no attribute 'GAME_ID'

In [21]:
len(set_a), len(set_b), len(set_b.difference(set_a))

(35346, 30701, 42)

In [23]:
season_flat

,GAME_ID,GAME_DATE,TEAM_ID,TEAM_ABBREVIATION,MATCHUP
0,0021901318,2020-08-14,1.610613e+09,DEN,DEN @ TOR
1,0021901317,2020-08-14,1.610613e+09,LAC,LAC vs. OKC
2,0021901316,2020-08-14,1.610613e+09,IND,IND vs. MIA
3,0021901316,2020-08-14,1.610613e+09,MIA,MIA @ IND
4,0021901315,2020-08-14,1.610613e+09,PHI,PHI @ HOU
...,...,...,...,...,...
70702,0021400002,2014-10-28,1.610613e+09,DAL,DAL @ SAS
70703,0021400001,2014-10-28,1.610613e+09,ORL,ORL @ NOP
70704,0021400002,2014-10-28,1.610613e+09,SAS,SAS vs. DAL
70705,0021400003,2014-10-28,1.610613e+09,LAL,LAL vs. HOU


In [3]:
n = disk.listdir('data/boxscores_v2/data')
q = disk.listdir('data/players_v2/data')
len(n), len(q)

(13610, 11804)

In [ ]:
gids = disk.read_json('data/unique_game_ids.json')
gids = gids[-1]['data']
gids = [_['game_id'] for _ in gids]

gids_check = map(lambda x: x.split('/')[-1], sorted(n, key=os.path.getctime))
gids_check = list(gids_check)
check = [i for i, (a, b) in enumerate(zip(gids_check, gids)) if a != b]

In [90]:
gids[:len(n)] == gids_check

True

In [ ]:
gids_check = map(lambda x: x.split('/')[-1], sorted(n, key=os.path.getctime))
gids_check = list(gids_check)
check = [i for i, (a, b) in enumerate(zip(gids_check, gids)) if a != b]

('0021600018', '0021600018')

In [95]:
import math

n = 10
k = 5

# Equivalent to n! / (k! * (n - k)!)
total_ways = math.comb(n, k)

print(total_ways)  # Output: 10


252
